# 06. Colab GPU용 PyTorch Target-Attention + DCN-v2 DeepCTR 5-Fold 학습 및 추론

이 노트북은 GBDT 트랙과 구조적 다양성(Model Diversity)을 확보하기 위해 **End-to-End PyTorch DeepCTR 모델(`SeqDCNv2CTRModel`)**을 학습하고 표준 OOF/Test 예측을 생성합니다.

### 핵심 설계 및 누수 방지 원칙
1. **Target-Attention + DCN-v2 하이브리드**: `seq` 토큰 임베딩(Tail Truncation `max_seq_len=100`, `PAD=0` 마스킹)에 광고/지면 컨텍스트(`inventory_id`, `l_feat_14`) 쿼리 기반 **DIN 스타일 Target Attention** + Mean/Last Pooling을 적용하고, 범주형 임베딩 및 `signed_log1p` 수치 피처(+ 결측 마스크)를 **CrossNetworkV2 + Deep MLP**로 결합합니다.
2. **폴드별 독립 전처리 (누수 0%)**: 모든 폴드마다 오직 해당 학습 폴드(`train_fold`)에서만 `FoldSeqVocabulary`, `FoldCategoricalIndexer`, `FoldNumericPreprocessor`를 `fit`하고 Memmap 스트리밍으로 변환해 Colab RAM 한계를 넘지 않습니다.
3. **`50:50 WLL` 직접 정렬 & AMP 가속**: `BCEWithLogitsLoss(pos_weight=N_0 / N_1)`와 `torch.amp.autocast('cuda')`를 사용해 GPU에서 고속 학습하며, `AP`·`WLL`·`Score`를 분리 평가합니다.

In [ ]:
from google.colab import drive
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import yaml

import os as _os
if not _os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# TODO: Google Drive 내 프로젝트 루트 경로에 맞게 수정하세요.
PROJECT_ROOT = Path('/content/drive/MyDrive/toss_ctr')
if (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if (PROJECT_ROOT / 'pyproject.toml').exists():
    os.chdir(PROJECT_ROOT)
    %pip install -e . --no-deps --quiet

print('PyTorch version:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / (1024 ** 3):.2f} GB')
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False).stdout)

from toss_ctr.data.splits import load_folds
from toss_ctr.evaluation.metrics import evaluate_predictions
from toss_ctr.features.seq_features import parse_seq_string
from toss_ctr.models.deep_ctr import (
    TORCH_AVAILABLE,
    infer_deep_feature_columns,
    load_deep_ctr_config,
    train_deep_ctr_oof_from_parquet,
    validate_deep_prediction_artifacts,
)

cfg_path = PROJECT_ROOT / 'configs' / 'deep_seq_dcn.yaml'
with open(cfg_path, 'r', encoding='utf-8') as f:
    raw_cfg = yaml.safe_load(f)
deep_cfg = load_deep_ctr_config(cfg_path)
print('TORCH_AVAILABLE:', TORCH_AVAILABLE)
print('Loaded DeepCTRConfig:', deep_cfg)

### [해석 1] Colab GPU/VRAM 자원 및 DeepCTR 설정 로드 확인
- CUDA GPU 가용 여부와 VRAM 용량을 점검해 `batch_size=2048` 및 `use_amp=True` FP16/BF16 혼합 정밀도 학습 환경을 확인했습니다.
- `configs/deep_seq_dcn.yaml`에서 `max_seq_len=100`, `min_freq=5`, `max_vocab_size=100000`, `seq_embed_dim=32`, `cat_embed_dim=16`, `cross_layers=3`, `mlp_dims=[256, 128, 64]`, `use_pos_weight_50_50=True` 설정이 정상 반영되었음을 확인합니다.

In [ ]:
# 입력 Data Contract 스키마 및 범주형/수치형 피처 분리 점검 (RAM 최소 사용)
paths_cfg = raw_cfg['paths']
train_tab_path = Path(paths_cfg['train_tabular'])
test_tab_path = Path(paths_cfg['test_tabular'])
train_seq_path = Path(paths_cfg['train_seq'])
test_seq_path = Path(paths_cfg['test_seq'])
folds_path = Path(paths_cfg['folds_path'])

tab_pf = pq.ParquetFile(train_tab_path)
seq_pf = pq.ParquetFile(train_seq_path)
test_tab_pf = pq.ParquetFile(test_tab_path)
test_seq_pf = pq.ParquetFile(test_seq_path)

n_train = tab_pf.metadata.num_rows
n_test = test_tab_pf.metadata.num_rows
assert n_train == seq_pf.metadata.num_rows, 'train_tabular과 train_seq 행 수 불일치!'
assert n_test == test_seq_pf.metadata.num_rows, 'test_tabular과 test_seq 행 수 불일치!'

folds_df = load_folds(folds_path, validate=True, n_rows=n_train, n_splits=deep_cfg.n_splits)
cat_cols, num_cols = infer_deep_feature_columns(tab_pf.schema_arrow)

y_preview = pq.read_table(train_tab_path, columns=['clicked']).column('clicked').to_numpy()
pos_cnt = int((y_preview == 1).sum())
neg_cnt = int((y_preview == 0).sum())
print(f'Train rows: {n_train:,} (Pos={pos_cnt:,}, Neg={neg_cnt:,}, CTR={pos_cnt/n_train:.4%})')
print(f'Expected 50:50 pos_weight (N_0 / N_1): {neg_cnt / pos_cnt:.4f}')
print(f'Test rows:  {n_test:,}')
print(f'범주형 피처 ({len(cat_cols)}개): {cat_cols}')
print(f'수치형 피처 ({len(num_cols)}개): {num_cols[:8]} ... (총 {len(num_cols)}개)')
print(f'Target Attention 쿼리 컬럼: {[c for c in deep_cfg.query_cols if c in cat_cols]}')

### [해석 2] Data Contract 무결성, 피처 분리 및 `50:50` 손실 가중치 사전 점검
- `train_tabular_diet`와 `train_seq`, `folds.parquet`의 행 수 및 `row_id`/`fold_id` 계약이 완벽히 일치함을 확인했습니다.
- 명시 범주형(`gender`, `age_group`, `inventory_id`, `day_of_week`, `hour`, `l_feat_14`)을 임베딩 + Target Attention 쿼리(`inventory_id`, `l_feat_14`)로 배정하고, 나머지 정형 피처는 `signed_log1p` + 폴드 내 표준화 + 결측 마스크 피처로 분리했습니다.
- 원천 CTR 불균형에 비례하는 `pos_weight = N_0 / N_1`을 적용함으로써 학습 목적함수가 리더보드 `50:50 WLL`과 수학적으로 정확히 정렬됩니다.

In [ ]:
# 5-Fold Memmap 스트리밍 + GPU AMP 학습 및 OOF/Test 추론 실행
pred_dir = Path(paths_cfg['pred_dir'])
oof_out_path = pred_dir / paths_cfg['oof_pred_filename']
test_out_path = pred_dir / paths_cfg['test_pred_filename']
model_dir = Path(paths_cfg['model_dir'])
report_path = Path(paths_cfg['report_dir']) / 'deep_seq_dcn_report.json'

t_start = time.perf_counter()
summary = train_deep_ctr_oof_from_parquet(
    train_tabular_path=train_tab_path,
    train_seq_path=train_seq_path,
    folds_path=folds_path,
    test_tabular_path=test_tab_path,
    test_seq_path=test_seq_path,
    config=deep_cfg,
    cat_cols=cat_cols,
    num_cols=num_cols,
    oof_output_path=oof_out_path,
    test_output_path=test_out_path,
    memmap_cache_dir=paths_cfg['memmap_cache_dir'],
    model_dir=model_dir,
    report_path=report_path,
    cleanup_cache=True,
)
elapsed = time.perf_counter() - t_start
print(f'5-Fold DeepCTR 학습·추론 완료! 총 소요 시간: {elapsed / 60:.1f}분')
print('Overall OOF Metrics:', summary['oof_metrics'])
print('OOT Val Metrics:    ', summary['oot_metrics'])

# 폴드별 Best 에포크 및 학습 곡선 시각화
fold_rows = []
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for fr in summary['fold_reports']:
    f_id = fr['fold_id']
    hist = pd.DataFrame(fr['history'])
    bm = fr['best_val_metrics']
    fold_rows.append({
        'fold_id': f_id,
        'best_epoch': fr['best_epoch'],
        'seq_vocab_size': fr['seq_vocab_size'],
        'pos_weight': fr['pos_weight'],
        'val_ap': bm['ap'],
        'val_wll': bm['wll'],
        'val_score': bm['score'],
    })
    axes[0].plot(hist['epoch'], hist['train_loss'], marker='o', label=f'Fold {f_id}')
    axes[1].plot(hist['epoch'], hist['val_ap'], marker='o', label=f'Fold {f_id}')
    axes[2].plot(hist['epoch'], hist['val_wll'], marker='o', label=f'Fold {f_id}')

axes[0].set_title('Train Weighted BCE Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].set_title('Validation AP (50%)')
axes[1].set_xlabel('Epoch')
axes[2].set_title('Validation 50:50 WLL (50%)')
axes[2].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

display(pd.DataFrame(fold_rows))

### [해석 3] 5-Fold 학습 곡선(Train Loss, Val AP, Val 50:50 WLL) 및 수렴 진단
- 5개 폴드 모두 학습 폴드 전용 `FoldSeqVocabulary`·`FoldCategoricalIndexer`·`FoldNumericPreprocessor`로 독립 전처리되었으며, Memmap 스트리밍 덕분에 Colab RAM 오버플로우 없이 학습을 완료했습니다.
- 에포크 진행에 따라 `train_loss`가 안정적으로 감소하고 검증 `AP` 상승 및 `50:50 WLL` 하락이 균형 있게 동반되는지 확인합니다. Early Stopping으로 과적합 이전의 최적 체크포인트가 복원되었습니다.

In [ ]:
# 시퀀스 길이 구간별 AP / 50:50 WLL / Score 및 확률 분포 진단
oof_df = pq.read_table(oof_out_path).to_pandas()
oof_df['is_oot_val'] = folds_df['is_oot_val'].to_numpy()

train_summary_p = Path(paths_cfg['train_seq_summary'])
if train_summary_p.exists():
    seq_len_arr = pq.read_table(train_summary_p, columns=['seq_token_len']).column(0).to_numpy()
else:
    # 05 노트북 산출물이 없는 경우에도 OOM 없이 길이만 스트리밍 산출
    lengths = []
    for batch in pq.ParquetFile(train_seq_path).iter_batches(batch_size=262144, columns=['seq']):
        lengths.extend(len(parse_seq_string(s, delimiter=deep_cfg.delimiter)) for s in batch.column(0).to_pylist())
    seq_len_arr = np.asarray(lengths, dtype=np.int32)

oof_df['seq_token_len'] = seq_len_arr
bins = [-1, 0, 10, 50, 100, 10_000_000]
labels = ['0 (Empty)', '1~10', '11~50', '51~100', '100+']
oof_df['len_bucket'] = pd.cut(oof_df['seq_token_len'], bins=bins, labels=labels)

bucket_evals = []
for b_label, grp in oof_df.groupby('len_bucket', observed=False):
    if len(grp) == 0 or grp['clicked'].nunique() < 2:
        continue
    ev = evaluate_predictions(grp['clicked'].to_numpy(), grp['pred_prob'].to_numpy())
    bucket_evals.append({
        'seq_len_bucket': str(b_label),
        'rows': len(grp),
        'share': round(len(grp) / len(oof_df), 4),
        'actual_ctr': round(float(grp['clicked'].mean()), 5),
        'mean_pred_prob': round(float(grp['pred_prob'].mean()), 4),
        'ap': round(ev.ap, 5),
        'wll': round(ev.wll, 5),
        'score': round(ev.score, 5),
    })

bucket_eval_df = pd.DataFrame(bucket_evals)
display(bucket_eval_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(bucket_eval_df['seq_len_bucket'], bucket_eval_df['ap'], color='steelblue')
axes[0].set_title('OOF AP by Sequence Length Bucket')
axes[0].set_ylabel('AP')
axes[1].bar(bucket_eval_df['seq_len_bucket'], bucket_eval_df['wll'], color='darkorange')
axes[1].set_title('OOF 50:50 WLL by Sequence Length Bucket')
axes[1].set_ylabel('WLL')
plt.tight_layout()
plt.show()

# OOF vs OOT 분리 성능 요약
oot_df = oof_df[oof_df['is_oot_val']]
print('=== Overall OOF vs OOT ===')
print('OOF:', evaluate_predictions(oof_df['clicked'].to_numpy(), oof_df['pred_prob'].to_numpy()).to_dict())
if len(oot_df) > 0 and oot_df['clicked'].nunique() == 2:
    print('OOT:', evaluate_predictions(oot_df['clicked'].to_numpy(), oot_df['pred_prob'].to_numpy()).to_dict())

### [해석 4] 시퀀스 길이 구간별 `AP`/`WLL` 분석 및 OOT 분포 일반화 검증
- **시퀀스 길이 구간별 변별력**: 빈 시퀀스(`0 (Empty)`) 구간과 유저 행동 이력이 풍부한(`11~50`, `51~100`, `100+`) 구간에서 `AP`와 `WLL`이 어떻게 개선되는지 확인해 Target Attention 및 Tail Truncation(`max_seq_len=100`)의 실질적 기여도를 검증합니다.
- **OOT 일관성**: 마지막 요일(`is_oot_val=True`) 구간과 전체 OOF 간의 `AP`/`WLL` 격차를 점검해 시간축 과적합이 없는지 확인합니다.

In [ ]:
# 표준 OOF / Test 예측 아티팩트 최종 무결성 검증 (ensemble-calibrator 인계 전 필수 확인)
expected_test_ids = pq.read_table(test_tab_path, columns=['ID']).column('ID').to_numpy()
check_res = validate_deep_prediction_artifacts(
    oof_path=oof_out_path,
    test_path=test_out_path,
    expected_n_train=n_train,
    expected_n_test=n_test,
    expected_test_ids=expected_test_ids,
)
print('Artifact Integrity Check:', check_res)

test_pred_df = pq.read_table(test_out_path).to_pandas()
display(oof_df[['row_id', 'fold_id', 'clicked', 'raw_logit', 'pred_prob']].head())
display(test_pred_df.head())
print(f'Test pred_prob summary: min={test_pred_df["pred_prob"].min():.5f}, '
      f'mean={test_pred_df["pred_prob"].mean():.5f}, max={test_pred_df["pred_prob"].max():.5f}')

### [해석 5] 표준 OOF / Test 아티팩트 스키마·정렬·확률 범위 무결성 확인
- `oof_deep_seq_dcn.parquet` (`row_id`, `fold_id`, `clicked`, `raw_logit`, `pred_prob`)가 `row_id` `0..N-1` 오름차순으로 완전히 채워졌고, `test_deep_seq_dcn.parquet` (`ID`, `raw_logit`, `pred_prob`)의 `ID` 순서가 원천 `test.parquet`와 100% 일치함을 검증했습니다.
- `50:50` 균형 손실로 학습되었기 때문에 `pred_prob` 평균이 `0.5` 부근의 보정된 분포를 형성하며 `ensemble-calibrator`의 Log-odds 블렌딩 입력으로 즉시 사용할 수 있습니다.

### [종합 요약 및 Gate 3/4 인계 안내]
- **모델 아키텍처**: `SeqDCNv2CTRModel` (DIN-style Target Attention + Mean/Last Pooling + `inventory_id`/`l_feat_14` Query + CrossNetworkV2 3층 + Deep MLP `[256, 128, 64]`)
- **생성된 표준 산출물**:
  - OOF 예측: `data/processed/predictions/oof_deep_seq_dcn.parquet`
  - Test 예측: `data/processed/predictions/test_deep_seq_dcn.parquet`
  - 폴드별 가중치 & 어휘집: `artifacts/deep_seq_dcn/`
  - 학습 리포트 JSON: `reports/deep_seq_dcn_report.json`
- **다음 단계**: Gate 3(`data-leakage-auditor` 감사) 통과 후 Gate 4(`ensemble-calibrator`)에서 GBDT 모델군(`LightGBM`/`CatBoost`/`XGBoost`)의 `raw_logit`과 본 DeepCTR `raw_logit`을 결합해 최종 `50:50 WLL` 캘리브레이션 및 제출 파일을 생성합니다.